# **Packages**

In [ ]:
!pip install transformers
!pip install torch

# **CLS Norm Method**

In [ ]:
# CLS METHOD
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"  # Avoid TensorFlow import issues

import torch
from transformers import BertTokenizer, BertModel
from collections import defaultdict
import nltk
import pandas as pd
import numpy as np
from google.colab import files
import re

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('english'))

# Upload file
uploaded = files.upload()
file_name = next(iter(uploaded.keys()))

# Read the file and take all reviews
with open(file_name, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Separate reviews and labels
reviews = []
labels = []
for line in lines:
    review, label = line.strip().rsplit('\t', 1)
    reviews.append(review)
    labels.append(int(label))

# Load BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_hidden_states=True)

# Function to calculate BERT score using [CLS] token embedding
def get_bert_score(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    hidden_states = outputs.hidden_states
    cls_embedding = hidden_states[-1][0][0]  # Extract the [CLS] token embedding from the last hidden layer
    score = torch.norm(cls_embedding).item()
    return score

# Split long reviews into chunks of max 512 tokens
def split_into_chunks(text, max_length=512):
    tokens = tokenizer.tokenize(text)
    chunks = [' '.join(tokens[i:i + max_length]) for i in range(0, len(tokens), max_length)]
    return chunks

# Clean stopwords and special tokens
def clean_tokens(tokens):
    return [
        token for token in tokens
        if token.lower() not in stop_words
        and not re.fullmatch(r'\W+', token)
        and token not in ['[PAD]', '[CLS]', '[SEP]']
    ]

# Dictionaries to store positive and negative sentences per word
positive_sentences = defaultdict(list)
negative_sentences = defaultdict(list)

def process_reviews(reviews, labels):
    for review, label in zip(reviews, labels):
        chunks = split_into_chunks(review.lower())
        for chunk in chunks:
            inputs = tokenizer(chunk, return_tensors='pt', truncation=True, padding=True)
            tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
            words = clean_tokens(tokens)
            for word in words:  # Include subwords
                if label == 1:
                    positive_sentences[word].append(review)
                else:
                    negative_sentences[word].append(review)

# Run processing
process_reviews(reviews, labels)

# Calculate term scores
term_scores = []

for word in set(list(positive_sentences.keys()) + list(negative_sentences.keys())):
    pos_score = None
    neg_score = None

    if word in positive_sentences:
        combined_positive_text = ' '.join(positive_sentences[word])
        pos_score = get_bert_score(combined_positive_text)

    if word in negative_sentences:
        combined_negative_text = ' '.join(negative_sentences[word])
        neg_score = get_bert_score(combined_negative_text)

    if pos_score is None:
        pos_score = 0.0
    if neg_score is None:
        neg_score = 0.0

    overall_score = np.log10((pos_score + 0.001) / (neg_score + 0.001))

    term_scores.append({
        'Term': word,
        'Positive Score': pos_score,
        'Negative Score': neg_score,
        'Overall Score': overall_score
    })

# Save to CSV
df = pd.DataFrame(term_scores)
df.to_csv('outputImdb.csv', index=False)
print('✅ Output saved to outputYelp.csv')


In [ ]:
# evaluation metrics
import torch
from transformers import BertTokenizer, BertModel
from collections import defaultdict
import nltk
import pandas as pd
from google.colab import files
import numpy as np
import math
from sklearn.metrics import precision_score, recall_score, f1_score

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('english'))

# Upload 'output.csv' file
print('Please upload the output.csv file')
output_file = files.upload()
output_file_name = next(iter(output_file.keys()))

# Load output.csv file
output_df = pd.read_csv(output_file_name)

# Create a dictionary of words and their overall scores from output.csv
word_scores = {}
for index, row in output_df.iterrows():
    word = row['Term']
    pos_score = row['Positive Score'] if not pd.isna(row['Positive Score']) else 0
    neg_score = row['Negative Score'] if not pd.isna(row['Negative Score']) else 0
    overall_score = math.log10((pos_score + 0.001) / (neg_score + 0.001))
    word_scores[word] = overall_score

# Upload 'amazon_cells_labelled.txt' file
print('Please upload the imdb.txt file')
review_file = files.upload()
review_file_name = next(iter(review_file.keys()))

# Read the first 1000 reviews
with open(review_file_name, 'r', encoding='utf-8') as f:
    lines = f.readlines()[:1000]

# Process the reviews and calculate their scores
review_data = []
true_labels = []
predicted_labels = []

for line in lines:
    review, label = line.strip().rsplit('\t', 1)
    label = 'Positive' if label == '1' else 'Negative'

    # Tokenize review
    tokens = review.lower().split()

    # Calculate overall score of the review
    overall_score = sum([word_scores.get(token, 0) for token in tokens])

    # Determine predicted sentiment
    predicted_sentiment = 'Positive' if overall_score > 0 else 'Negative'

    # Check if prediction matches actual sentiment
    match = 1 if predicted_sentiment == label else 0

    review_data.append([review, label, overall_score, predicted_sentiment, match])
    true_labels.append(1 if label == 'Positive' else 0)
    predicted_labels.append(1 if predicted_sentiment == 'Positive' else 0)

# Create DataFrame and save to CSV
review_score_df = pd.DataFrame(review_data, columns=['Review', 'Sentiment', 'Overall Score', 'Predicted Sentiment', 'Match'])
review_score_df.to_csv('review_scores_Imdb.csv', index=False)

# Calculate accuracy
accuracy = review_score_df['Match'].sum() / len(review_score_df)

# Calculate Precision, Recall, F1 Score
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)

# Save evaluation metrics to CSV
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})
metrics_df.to_csv('evaluation_metrics_Imdb.csv', index=False)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print('Review scores saved to review_scores_Imdb.csv')
print('Evaluation metrics saved to evaluation_metrics_Imdb.csv')


# **WORD LEVEL SENTIMENT SCORES USING AVERAGED BERT EMBEDDINGS**

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import pandas as pd
from google.colab import files
import io
from collections import defaultdict
from nltk.corpus import stopwords
import nltk
import numpy as np
import math

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Upload the dataset
print("Please upload the labeled dataset (e.g., amazon_cells_labelled.txt):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load dataset
df = pd.read_csv(io.BytesIO(uploaded[file_name]), delimiter='\t', header=None, names=['review', 'sentiment'])
df = df.head(1000)  # Limit to first 100 reviews

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

# Store word-level scores
positive_embeddings = defaultdict(list)
negative_embeddings = defaultdict(list)

# Process reviews
for _, row in df.iterrows():
    text = row['review']
    label = row['sentiment']

    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = torch.stack(outputs.hidden_states)  # (13 layers, 1, seq_len, hidden_size)
    token_embeddings = hidden_states.mean(dim=0).squeeze(0)  # Average across all 12 layers

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    for token, embedding in zip(tokens, token_embeddings):
        if token in stop_words or token.startswith('##') or token in tokenizer.all_special_tokens:
            continue
        score = abs(embedding.mean().item())  # Ensures strictly positive scores
        if label == 1:
            positive_embeddings[token].append(score)
        else:
            negative_embeddings[token].append(score)

# Combine results
words = set(list(positive_embeddings.keys()) + list(negative_embeddings.keys()))
results = []

for word in words:
    pos_score = np.mean(positive_embeddings[word]) if word in positive_embeddings else 0.0
    neg_score = np.mean(negative_embeddings[word]) if word in negative_embeddings else 0.0

    # Compute overall score safely with +0.01 in formula
    overall_score = math.log10((abs(pos_score) + 0.01) / (abs(neg_score) + 0.01))

    results.append({
        'Word': word,
        'Positive Score': round(pos_score, 5),
        'Negative Score': round(neg_score, 5),
        'Overall Score': round(overall_score, 5)
    })

# Save to CSV
output_df = pd.DataFrame(results)
output_file = "output_Imdb_averagemethod.csv"
output_df.to_csv(output_file, index=False)
print(f"\nWord scores saved to {output_file}")
files.download(output_file)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import files
import io

# Step 1: Upload the review dataset
print("Please upload the labeled review dataset (e.g., imdb_labelled.txt):")
uploaded_review = files.upload()
review_file = list(uploaded_review.keys())[0]
df_reviews = pd.read_csv(io.BytesIO(uploaded_review[review_file]), delimiter='\t', header=None, names=['Review', 'Sentiment'])

# Step 2: Upload the word score CSV file
print("Please upload the averaged word score file (e.g., averaged_word_scores.csv):")
uploaded_scores = files.upload()
score_file = list(uploaded_scores.keys())[0]
df_scores = pd.read_csv(io.BytesIO(uploaded_scores[score_file]))

# Build word -> overall score dictionary
word_scores = dict(zip(df_scores['Word'], df_scores['Overall Score']))

# Compute score for each review
results = []

for _, row in df_reviews.iterrows():
    review = row['Review']
    sentiment = row['Sentiment']

    words = review.lower().split()
    score = sum([word_scores.get(word, 0.0) for word in words])

    predicted = 1 if score > 0 else 0
    match = 1 if predicted == sentiment else 0

    results.append({
        'Review': review,
        'Sentiment': sentiment,
        'Score': round(score, 4),
        'Predicted Sentiment': predicted,
        'Match': match
    })

# Save prediction results
df_result = pd.DataFrame(results)
result_file = "review_predictions_imdb_average.csv"
df_result.to_csv(result_file, index=False)
print(f"\nPredictions saved to {result_file}")
files.download(result_file)

# Compute and save evaluation metrics
y_true = df_result['Sentiment']
y_pred = df_result['Predicted Sentiment']

metrics = {
    'Accuracy': [round(accuracy_score(y_true, y_pred), 4)],
    'Precision': [round(precision_score(y_true, y_pred), 4)],
    'Recall': [round(recall_score(y_true, y_pred), 4)],
    'F1 Score': [round(f1_score(y_true, y_pred), 4)]
}

df_metrics = pd.DataFrame(metrics)
metrics_file = "evaluation_metrics_imdb_average.csv"
df_metrics.to_csv(metrics_file, index=False)
print(f"Evaluation metrics saved to {metrics_file}")
files.download(metrics_file)


# **ZERO SHOT LEXICON BASED MENTHODS**

In [ ]:
# ZERO_SHOT LEXICON BASED SENTIMENT ANALYSIS

# ✅ Step 1: Imports
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import files

# ✅ Step 2: Load model
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# ✅ Step 3: Upload input file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ Step 4: Load and sample 1000 reviews
with open(filename, 'r', encoding='utf-8') as f:
    lines = f.readlines()

reviews = [line.strip().split('\t') for line in lines if len(line.strip().split('\t')) == 2][:1000]

# ✅ Step 5: Build word sentiment score dictionary
word_scores = {}

for review, sentiment in tqdm(reviews, desc="Processing words"):
    words = word_tokenize(review.lower())
    for word in words:
        if not re.match(r"^[a-zA-Z0-9]+$", word):
            continue
        if word in word_scores:
            continue
        inputs = tokenizer(word, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1).squeeze().tolist()
        pos_score = probs[2]
        neg_score = probs[0]
        word_scores[word] = {
            "positive": round(pos_score, 4),
            "negative": -round(neg_score, 4)
        }

# ✅ Step 6: Analyze reviews and calculate scores
review_scores_data = []
prediction_data = []

for review, sentiment in tqdm(reviews, desc="Scoring reviews"):
    words = word_tokenize(review.lower())
    scores = []
    for word in words:
        if not re.match(r"^[a-zA-Z0-9]+$", word):
            continue
        if word in word_scores:
            pos = word_scores[word]["positive"]
            neg = word_scores[word]["negative"]
            score = pos + neg
            scores.append((word, score))
        else:
            scores.append((word, 0.0))

    score_expr = " + ".join([f"{s:.4f}" for _, s in scores])
    total = sum(s for _, s in scores)
    formula = total / len(scores) if scores else 0
    predicted = "positive" if formula > 0 else "negative"
    actual_sentiment = "positive" if sentiment == "1" else "negative"
    match = int(predicted == actual_sentiment)

    review_scores_data.append({
        "Review": review,
        "Sentiment": actual_sentiment,
        "Score Expression": score_expr,
        "Sum": round(total, 4),
        "Formula (Sum/Count)": round(formula, 4),
        "Predicted Sentiment": predicted,
        "Match": match
    })

    prediction_data.append({
        "Review": review,
        "Actual Sentiment": actual_sentiment,
        "Predicted Sentiment": predicted,
        "Match": match
    })

# ✅ Step 7: Save review scores and predictions
df_scores = pd.DataFrame(review_scores_data)
df_preds = pd.DataFrame(prediction_data)

df_scores.to_csv("review_scores.csv", index=False)
df_preds.to_csv("predictions.csv", index=False)

files.download("review_scores.csv")
files.download("predictions.csv")

# ✅ Step 8: Compute and save evaluation metrics
y_true = [1 if d["Sentiment"] == "positive" else 0 for d in review_scores_data]
y_pred = [1 if d["Predicted Sentiment"] == "positive" else 0 for d in review_scores_data]

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

metrics = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [round(acc, 4), round(prec, 4), round(rec, 4), round(f1, 4)]
}

df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("metrics.csv", index=False)
files.download("metrics.csv")

# ✅ Optional: Print metrics in output
print("\n📊 Evaluation Metrics:")
for metric, value in zip(metrics["Metric"], metrics["Value"]):
    print(f"{metric}: {value}")


# **AFINN**

In [ ]:
# AFINN

# ✅ Step 1: Install & Imports
!pip install afinn -q
import pandas as pd
import numpy as np
from afinn import Afinn
from nltk.tokenize import word_tokenize
import nltk
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import files
import re

nltk.download('punkt')
nltk.download('punkt_tab')

afinn = Afinn()

# ✅ Step 2: Upload File
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ Step 3: Read reviews (tab-separated)
with open(filename, 'r', encoding='utf-8') as f:
    lines = f.readlines()

reviews = [line.strip().split('\t') for line in lines if len(line.strip().split('\t')) == 2][:1000]

# ✅ Step 4: Process reviews
output_data = []
results = []

for review, sentiment in tqdm(reviews, desc="Scoring reviews"):
    words = word_tokenize(review.lower())
    filtered = [word for word in words if re.match(r"^[a-zA-Z0-9]+$", word)]

    word_scores = [(word, afinn.score(word)) for word in filtered]
    score_expr = " + ".join([f"{score:.2f}" for _, score in word_scores])
    total = sum(score for _, score in word_scores)
    avg_score = total / len(word_scores) if word_scores else 0
    predicted = "positive" if avg_score > 0 else "negative"
    actual = "positive" if sentiment == "1" else "negative"
    match = int(predicted == actual)

    # Detailed review scores
    output_data.append({
        "Review": review,
        "Sentiment": actual,
        "Score Expression": score_expr,
        "Sum": round(total, 4),
        "Formula (Sum/Count)": round(avg_score, 4),
    })

    # Prediction & match
    results.append({
        "Review": review,
        "Actual Sentiment": actual,
        "Predicted Sentiment": predicted,
        "Match": match
    })

# ✅ Step 5: Save detailed scores
df_scores = pd.DataFrame(output_data)
df_scores.to_csv("afinn_review_scores.csv", index=False)
files.download("afinn_review_scores.csv")

# ✅ Step 6: Save predictions
df_results = pd.DataFrame(results)
df_results.to_csv("afinn_predictions.csv", index=False)
files.download("afinn_predictions.csv")

# ✅ Step 7: Evaluation metrics
y_true = [1 if row["Actual Sentiment"] == "positive" else 0 for row in results]
y_pred = [1 if row["Predicted Sentiment"] == "positive" else 0 for row in results]

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

metrics_df = pd.DataFrame([{
    "Accuracy": round(acc, 4),
    "Precision": round(prec, 4),
    "Recall": round(rec, 4),
    "F1 Score": round(f1, 4)
}])

metrics_df.to_csv("afinn_metrics.csv", index=False)
files.download("afinn_metrics.csv")

# ✅ Show in console too
print("\n📊 Evaluation Metrics:")
print(metrics_df.to_string(index=False))
